In [6]:
from Bio import SeqIO
import pandas as pd
import random

# for loading features
def load_dict(path):
    res = {}
    lines = open(path).readlines()
    for line in lines:
        x_list = line.strip().split(",")
        name = x_list[0]
        vec = [float(x) for x in x_list[1:]]
        res[name]=vec
    return res

# for loading size
def read_fa(path):
    res={}
    rescords = list(SeqIO.parse(path,format="fasta"))
    for x in rescords:
        id = str(x.id)
        seq = str(x.seq).replace("U","T")
        res[id]=seq
    return res

In [7]:
lnc_kmer_dict = load_dict("./../data/kmers/lnc_kmer_dict.txt")
mir_kmer_dict = load_dict("./../data/kmers/mir_kmer_dict.txt")
ss_loops_dict = load_dict("./../data/binding_features_dic.txt")
lnc_loops_dict = load_dict("./../data/lnc_features.csv")

In [8]:
mirna_dict = read_fa("./../data/fasta/fasta_seq/miRNA.fa")
lncrna_dict = read_fa("./../data/fasta/fasta_seq/lncRNA.fa")

In [9]:
negative_pairs_path = "./../data/txt_interac/negative_pairs.txt"
positive_pairs_path = "./../data/txt_interac/mirnas_lncrnas_validated_positive_pairs.txt"

positive_pairs = [[line.strip().split(",")[0],line.strip().split(",")[1]] for line in open(positive_pairs_path,"r").readlines()]
negative_pairs = [[line.strip().split(",")[0],line.strip().split(",")[1]] for line in open(negative_pairs_path,"r").readlines()]

all_pairs = positive_pairs + negative_pairs
rows = []
filters = ["miR-17", "miR-19a", "miR-20a"]

for pair in all_pairs:
    if any(mir_specific in pair[1] for mir_specific in filters):
        lnc = pair[0]
        mir = pair[1]
        lnc_vec = lnc_kmer_dict[lnc]
        mir_vec = mir_kmer_dict[mir]
        binding_feature_list = ss_loops_dict[f'{lnc}_{mir}']
        lnc_features = lnc_loops_dict[lnc]
        lnc_len = len(lncrna_dict[lnc])
        mir_len = len(mirna_dict[mir])
        row = [mir_len, lnc_len] + lnc_features + binding_feature_list + mir_vec + lnc_vec
        rows.append(row)

# filtered_negatives = [pair for pair in negative_pairs if not any(mir_specific in pair[1] for mir_specific in filters)]
# missing_negatives = random.sample(filtered_negatives, 166)

# for pair in missing_negatives:
#     lnc = pair[0]
#     mir = pair[1]
#     lnc_vec = lnc_kmer_dict[lnc]
#     mir_vec = mir_kmer_dict[mir]
#     binding_feature_list = ss_loops_dict[f'{lnc}_{mir}']
#     # Uncomment next line to add lncRNA features.
#     lnc_features = lnc_loops_dict[lnc]
#     lnc_len = len(lncrna_dict[lnc])
#     mir_len = len(mirna_dict[mir])
#     # Uncomment next line and comment the one after to add lncRNA features.
#     row = [mir_len, lnc_len] + lnc_features + binding_feature_list + mir_vec + lnc_vec + pair[2]
#     # row = [mir_len, lnc_len] + binding_feature_list + mir_vec + lnc_vec
#     rows.append(row)

df = pd.DataFrame(rows)

In [10]:
df.to_csv("./../data/filtered_input_train_lnc_features.csv", index=False, header=False)